# Surrogate evaluation

This notebook evaluates a trained surrogate `SurrogateLSTM` on the test split. The model is the checkpoint that `surrogate_training.ipynb` saves: the weights of the epoch with the lowest validation error. The test split is the same as in training, because `DatasetBuilder` splits the V/T groups with a fixed seed.

Steps:

1. **Setup and settings**: libraries, device (GPU/CPU), dataset, checkpoint and spike tolerance in one place.
2. **Data**: the test split from `DatasetBuilder`.
3. **Evaluation**: error on the test split, example curves, and the spikes: hits, missed and extra spikes, timing and height.

Requirements: a kernel with the packages from `VAT_SNN/requirements.txt` (torch, numpy, matplotlib). The notebook must run from its folder `surrogate/notebooks` so that the paths are right. `DATASET_NAME` and `CHECKPOINT_PATH` must be the same as in the training run.

## 1. Setup and settings

In [ ]:
# standard library: paths
import sys
from pathlib import Path

# arrays and plots
import numpy as np
import matplotlib.pyplot as plt
# tensors
import torch

# folder surrogate/src with our code (the notebook runs in surrogate/notebooks)
SRC_DIR = Path.cwd().parent / "src"
# put it at the front of the import path so that "import model" finds our file
sys.path.insert(0, str(SRC_DIR))

# our own code: raw data → DataLoaders, the model, and the spike comparison
from dataset_builder import DatasetBuilder
from model import SurrogateLSTM
from metrics import compare_spikes

# evaluate on the GPU if there is one, otherwise on the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# show which device is used
print("device:", device)

In [ ]:
# dataset folder inside surrogate/cadence/datasets (the one the model was trained on)
DATASET_NAME = "surrogate_ds_v1"
# time between two curve points after downsampling (DatasetBuilder default step=10 on the 1 µs grid)
DT = 10e-6
# curves per mini-batch when predicting (only changes speed and memory, not the result)
BATCH_SIZE = 64
# spike comparison: largest distance between the starts of a SPICE and a surrogate spike to count as a pair (3 × 10 µs = 30 µs)
TOLERANCE_STEPS = 3
# saved model from surrogate_training.ipynb (CHECKPOINT_PATH there)
CHECKPOINT_PATH = Path.cwd().parent / "checkpoints" / "surrogate_lstm_van_rossum.pt"

## 2. Data

`DatasetBuilder` reads the raw curves and splits them 70/20/10 by V/T group, exactly as in the training notebook (same seed, therefore the same test groups). Only the test loader is used here.

In [ ]:
# read the raw curves, split them 70/20/10 by V/T group and wrap every split into a DataLoader; keep only the test loader
test_loader = DatasetBuilder(DATASET_NAME).build_loaders(BATCH_SIZE).test_loader
# print the number of test curves (the loader holds its SurrogateDataset in test_loader.dataset)
print(f"test: {len(test_loader.dataset)} curves")

## 3. Evaluation on the test split

For the evaluation we load the saved best model and predict all test curves. The test groups have V/T combinations that occurred neither in training nor in validation.

We look at three things:

1. **Voltage error**: RMSE over all test curves, plus median and maximum per curve.
2. **Example curves**: the best, a median and the worst test curve compared with SPICE.
3. **Spikes**: for the P-SNN what matters in the end is whether the neuron fires at the right time. As during training, `compare_spikes` pairs every SPICE spike with at most one surrogate spike that starts at most `TOLERANCE_STEPS` steps away. We count the pairs (hits), the missed and the extra spikes. For the hits we also look at the timing (start of the surrogate spike minus start of the SPICE spike) and at the height (peak of the surrogate spike minus peak of the SPICE spike).

In [ ]:
# load the weights of the epoch with the lowest validation error
checkpoint = torch.load(CHECKPOINT_PATH)
# rebuild the model with the saved size and load the weights
model = SurrogateLSTM(**checkpoint["config"]).to(device)
model.load_state_dict(checkpoint["model"])
# eval mode: dropout off
model.eval()
# show which epoch was the best
print("best model from epoch", checkpoint["epoch"])

# no gradients needed for the evaluation
with torch.no_grad():
    # predict every test batch, move it back to the CPU and join all batches
    test_prediction = torch.cat([model(*[tensor.to(device) for tensor in batch[:5]]).cpu() for batch in test_loader]).numpy()

# test split (the SurrogateDataset behind the test loader) as numpy arrays
test = test_loader.dataset
test_input = test.input_voltage.numpy()
test_target = test.output_voltage.numpy()
test_vdd = test.vdd.numpy()
test_temperature = test.temperature.numpy()

# squared error of every value
squared_error = (test_prediction - test_target) ** 2
# RMSE of every curve
rmse_per_curve = np.sqrt(squared_error.mean(axis=1))
# RMSE over all test values
print(f"test RMSE overall: {np.sqrt(squared_error.mean()) * 1000:.1f} mV")
# typical and worst curve
print(f"per curve: median {np.median(rmse_per_curve) * 1000:.1f} mV | worst {rmse_per_curve.max() * 1000:.1f} mV")

In [ ]:
# curves sorted from the smallest to the largest error
order = np.argsort(rmse_per_curve)
# best, median and worst test curve
examples = {"best": order[0], "median": order[len(order) // 2], "worst": order[-1]}
# time axis in ms
time_ms = np.arange(test_target.shape[1]) * DT * 1000

# one row per example curve, shared time axis
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
# draw every example into its own row
for ax, (label, i) in zip(axes, examples.items()):
    # input voltage in grey in the background
    ax.plot(time_ms, test_input[i], color="grey", alpha=0.5, label="vg1 (input)")
    # SPICE output and surrogate output
    ax.plot(time_ms, test_target[i], label="vout SPICE")
    ax.plot(time_ms, test_prediction[i], "--", label="vout surrogate")
    # title with error and operating point of this curve
    ax.set_title(f"{label} curve: RMSE {rmse_per_curve[i] * 1000:.1f} mV | VDD {test_vdd[i]:.3f} V | T {test_temperature[i]:.1f} °C")
    # y axis label
    ax.set_ylabel("voltage [V]")
# legend only once
axes[0].legend(loc="upper right")
# x axis label only at the bottom
axes[-1].set_xlabel("time [ms]")
# avoid overlapping titles
plt.tight_layout()
# show the plot
plt.show()

In [ ]:
# pair the spikes of surrogate and SPICE in all test curves
test_spikes = compare_spikes(test_prediction, test_target, test_vdd, TOLERANCE_STEPS)
# timing error of every hit in µs (positive: surrogate too late)
timing_error = test_spikes.timing_error * DT * 1e6
# peak error of every hit in mV (negative: surrogate too low)
peak_error = test_spikes.peak_error * 1000

# number of spikes on both sides
print(f"spikes: SPICE {test_spikes.n_true} | surrogate {test_spikes.n_predicted}")
# hits, missed and extra spikes, each also as share of the SPICE spikes
print(f"hits (at most {TOLERANCE_STEPS * DT * 1e6:.0f} µs apart): {test_spikes.n_hits} ({test_spikes.n_hits / test_spikes.n_true * 100:.1f} %) | "
      f"missed: {test_spikes.n_missed} ({test_spikes.n_missed / test_spikes.n_true * 100:.1f} %) | "
      f"extra: {test_spikes.n_extra} ({test_spikes.n_extra / test_spikes.n_true * 100:.1f} %) of the SPICE spikes")
# spike F1 score
print(f"spike F1 score: {test_spikes.f1:.3f}")
# timing and height, only if there are hits at all
if test_spikes.n_hits > 0:
    # typical and large timing error, and the mean shift (shows a constant delay)
    print(f"timing of the hits: median |Δt| {np.median(np.abs(timing_error)):.0f} µs | 95th percentile {np.percentile(np.abs(timing_error), 95):.0f} µs | "
          f"mean shift {timing_error.mean():+.1f} µs (time grid {DT * 1e6:.0f} µs)")
    # typical and large height error, and the mean shift (negative: spikes too low)
    print(f"peak height of the hits: median |Δ| {np.median(np.abs(peak_error)):.0f} mV | 95th percentile {np.percentile(np.abs(peak_error), 95):.0f} mV | "
          f"mean shift {peak_error.mean():+.0f} mV")